In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre_processing

### 1. Coverting to Lower_Case

In [8]:
df["review"]=df["review"].str.lower()

### 2. Removing the URLS

In [9]:
import re
def remove_urls(text):
    if isinstance(text, str):
        return re.sub(r"http\S+", "", text)
    return text  

df["review"] = df["review"].apply(remove_urls)


### 3. Removing Punctuations

In [10]:
import re

def remove_punctuations(text):
    if isinstance(text, str):  # handle None/NaN safely
        # Keep only letters, digits, and spaces; remove everything else
        return re.sub(r"[^A-Za-z0-9\s]", "", text)
    return ""
    
df["review"] = df["review"].apply(remove_punctuations)


### 4. Removing HTML

In [11]:
import re

def remove_html(text):
    if isinstance(text, str):  # handle None/NaN safely
        # Keep only letters, digits, and spaces; remove everything else
        return re.sub(r"<.*?>", "", text)
    return ""
    
df["review"] = df["review"].apply(remove_html)

### 5. Removing Stopwords

In [19]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Piyush
[nltk_data]     Gupta\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\Piyush
[nltk_data]     Gupta\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Piyush
[nltk_data]     Gupta\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
    token=word_tokenize(text)
    stop_words= stopwords.words("English")

    for word in token:
        if word in stop_words:
            text=text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [14]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [15]:
# running-> run
# played->play

from nltk.stem import PorterStemmer

In [16]:
def stemming(text):
    ps= PorterStemmer()
    stemmed_words=[]
    tokens= word_tokenize(text)

    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


### 7. Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [19]:
y=df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

x=tf.fit_transform(df["review"])

In [22]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3223792 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 4935)	0.1615456982175544
  (0, 3423)	0.5076906829597388
  (0, 1454)	0.1800447023658685
  (0, 3972)	0.16423069238821728
  (0, 1598)	0.10378653980387573
  (0, 2317)	0.0812687712868407
  (0, 257)	0.08898371800829066
  (0, 1920)	0.1830699881292457
  (0, 4291)	0.06509483889312083
  (0, 4155)	0.0691113283732711
  (0, 4700)	0.29937880691858304
  (0, 4194)	0.09532563061174032
  (0, 2050)	0.037753903268788636
  (0, 4205)	0.2300198649406693
  (0, 1925)	0.0860519053865076
  (0, 4196)	0.08707637965718773
  (0, 4591)	0.0827762776924278
  (0, 4722)	0.15494599652724345
  (0, 402)	0.06944845772327474
  (0, 2143)	0.062119461197249176
  (0, 4956)	0.10250793208256041
  (0, 1483)	0.05840776295077719
  (0, 3689)	0.11340958785019382
  (0, 456)	0.09802600042025161
  (0, 2405)	0.03046794582625496
  :	:
  (49581, 249)	0.08149262172993556
  (49581, 4110)	0.09005738363138398
  (495

## DataSet and DataLoader

In [36]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test= train_test_split(x, y, test_size=0.2, random_state=42)


In [37]:
x_train.shape

(39665, 5000)

In [42]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# If they are already NumPy arrays
x_train = x_train
x_test = x_test


In [43]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)
    

In [44]:
train_loader=DataLoader(train_set, shuffle=True, batch_size=64)
test_loader=DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [45]:
import torch.nn as nn
import torch.optim as optim

In [46]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super(RNN, self). __init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        # RNN_Layer
        self.rnn=nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully_Connected_Layer
        self.fc= nn.Linear(hidden_size,1)

        
    def forward(self,x):
        h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _=self.rnn(x,h0)
        # 1st value= hidden state of all the timesteps -> (batch, seq_len, hidden_size)
        # 2nd value= final hidden state of last timestep

        out=self.fc(out[:, -1, :])
        return out

In [47]:
input_size= x_train.shape[1]

model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [48]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,1
1,wderful ltle prducti br br filming technique...,1
2,hugh h werful w pen me h ummer weeken n...,1
3,bcy fly e boy jke hk zobe cloe pn f...,0
4,peer me love i vu unng film wch mr mei ...,1


## Training the RNN

In [49]:
epochs=10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1)  #add singleton direction

        outputs=model(xb) # (batch_size,1)
        outputs=torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss= criterion(outputs,yb) #compute loss
        loss.backward() #backpropogation
        optimizer.step() #weights update
    print(f"epoch={epoch+1}/{epochs} and loss= {loss.item()}")

epoch=1/10 and loss= 0.3148719370365143
epoch=2/10 and loss= 0.240938201546669
epoch=3/10 and loss= 0.2755948603153229
epoch=4/10 and loss= 0.3676254153251648
epoch=5/10 and loss= 0.2336178570985794
epoch=6/10 and loss= 0.33396467566490173
epoch=7/10 and loss= 0.16667529940605164
epoch=8/10 and loss= 0.2799082100391388
epoch=9/10 and loss= 0.2723744809627533
epoch=10/10 and loss= 0.3742409646511078


## Evaluate

In [50]:
model.eval()

with torch.no_grad():
    correct_vals=0
    tot_vals=0

    for xb, yb in test_loader:
        xb=xb.unsqueeze(1)

        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals+= yb.size(0)
        correct_vals+= (predicted==yb).sum().item()
    print(f"Accuracy: {correct_vals/tot_vals * 100}")

Accuracy: 82.58545931229202
